# 00 — Why Normal Language Models Fail (and How RAG Fixes It)

**By the end of this notebook, you can explain why standalone LLMs hallucinate on private or recent facts and prove how supplying retrieved context fixes it.**

### Learning Objectives
- Observe a standalone LLM failing on private company policies and October 2026 release facts.
- Compare prompt-stuffing, fine-tuning, and RAG across cost, freshness, and factual recall.
- Formulate a crisp one-sentence definition of Retrieval-Augmented Generation (RAG).

### Prerequisites
- None — basic Python is all you need.

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 01:26:07.011000 1420 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: Closed-Book Guessing vs. Open-Book Evidence

A **Large Language Model (LLM)** memorizes general knowledge during training — like a student taking a **closed-book exam**. When asked about *private company documents* or events *after its training cutoff*, the model cannot check external files, leading to **hallucination** (confident invention) or polite refusal.

Pasting every manual into the prompt hits a strict **context window** limit (like trying to spread 5,000 pages across a tiny school desk), while **fine-tuning** (re-training weights) costs compute and excels at style/formatting rather than mutable exact facts.

| Approach | Freshness (2026 Updates) | Private Facts Recall | Context Cost | Best Used For |
|---|---|---|---|---|
| **Prompt-Stuffing Whole Files** | Instant | High (small files only) | Very High (fails on 1,000+ pages) | Tiny single-page docs |
| **Fine-Tuning** | Stale after training | Poor / brittle exact recall | High training compute | Domain tone & output style |
| **RAG** | Instant upon index update | High with inline citations | Low (top 2–4 snippets only) | Up-to-date & private Q&A |

```
Closed-Book: [Question] --> (Frozen LLM Weights) --> Guess / Refusal

Open-Book  : [Question] --> [Find Top Snippets] --> (Context + Question) --> [LLM] --> Grounded Answer
```

## 3. Step-by-Step Build

### Step 1: Ask the LLM About Novasphere WITHOUT Context
Let's ask our model an internal 2026 question about our fictional company Novasphere Dynamics before providing any documents.

In [2]:
question = "How much does Novasphere Dynamics charge per month for the Enterprise Plan?"
print("Closed-book output:", llm.invoke(question).content)
        

Closed-book output: [OfflineDemoChatModel] I cannot verify proprietary 2026 Novasphere Dynamics facts without retrieved documents. Supply context via RAG.


> **What just happened?**
> Without retrieved documents, the model explicitly cannot verify private company pricing.

### Step 2: Paste the ENTIRE Handbook Into the Prompt (Prompt-Stuffing)
For one small text file, pasting the full document into `<context>` gives the model everything it needs to answer accurately.

In [3]:
handbook = (data_dir / "company_handbook.txt").read_text(encoding="utf-8")
stuffed_prompt = f"<context>\n{handbook}\n</context>\n\nQuestion: {question}"
print("Stuffed chars:", len(handbook))
print("Open-book reply:", llm.invoke(stuffed_prompt).content)
        

Stuffed chars: 2897
Open-book reply: Based on the provided context: 4. COMMERCIAL PRICING PLANS (NOVASPHERE OS) All commercial subscriptions include automated over-the-air (OTA) firmware signing and AES-256 encrypted telemetry: Starter Plan: $499 per month. Supports up to 10 active robot or drone nodes. Includes standard dashboards and 30-day log retention. Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retention, and 24/7/365 mission-critical support. NOVASPHERE DYNAMICS — EMPLOYEE & OPERATIONS HANDBOOK (2026 Edition)


> **What just happened?**
> Prompt-stuffing worked for a 2 KB file — now it extracted `$4,200 per month` cleanly.

### Step 3: See Why Prompt-Stuffing Breaks at Real-World Scale
Let's measure token estimates (~4 chars per token) if a real company has 5,000 such handbooks instead of one tiny demo file.

In [4]:
approx_tokens = len(handbook) // 4
tokens_5000_docs = approx_tokens * 5000
print(f"1 handbook ~ {approx_tokens:,} tokens | 5,000 docs ~ {tokens_5000_docs:,} tokens (overflows context & spikes latency/cost!)")
        

1 handbook ~ 724 tokens | 5,000 docs ~ 3,620,000 tokens (overflows context & spikes latency/cost!)


> **What just happened?**
> At 5,000 documents (~3M+ tokens), stuffing exceeds model limits, increases cost 5,000x, and buries the needle in a haystack.

### Step 4: Select Only the Matching Snippet (The Core Idea Behind RAG)
If we extract ONLY the relevant pricing paragraph first, token cost plummets while answer accuracy stays crisp!

In [5]:
snippet = "\n".join(l for l in handbook.splitlines() if "Plan:" in l)
mini_rag_prompt = f"<context>\n{snippet}\n</context>\n\nQuestion: {question}"
print(f"Focused snippet tokens: ~{len(snippet)//4} | Answer: {llm.invoke(mini_rag_prompt).content}")
        

Focused snippet tokens: ~124 | Answer: Based on the provided context: Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retention, and 24/7/365 mission-critical support. Starter Plan: $499 per month. Supports up to 10 active robot or drone nodes. Includes standard dashboards and 30-day log retention. Professional Plan: $1,850 per month. Supports up to 50 active nodes. Includes the HyperSync protocol, 90-day telemetry retention, and dedicated Slack Connect support.


> **What just happened?**
> That one-sentence essence: **RAG retrieves only the most relevant snippets from external data at query time so the LLM generates a grounded answer.**

## 4. Try It Yourself
1. **Test a recent 2026 release** — *Hint:* Compare asking about `Dr. Elina Vance` without vs. with `recent_changelog.md`.
2. **Try an unanswerable gap** — *Hint:* Ask for Novasphere's NASDAQ stock ticker using `recent_changelog.md` as context.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [6]:
changelog = (data_dir / "recent_changelog.md").read_text(encoding="utf-8")
print("No context:", llm.invoke("Who was appointed Chief AI Ethics Officer?").content)
print("With docs :", llm.invoke(f"<context>\n{changelog}\n</context>\nQuestion: Who was appointed Chief AI Ethics Officer?").content)
print("Missing   :", llm.invoke(f"<context>\n{changelog}\n</context>\nQuestion: What is Novasphere's NASDAQ stock ticker?").content)
        

No context: [OfflineDemoChatModel] Ready.
With docs : Based on the provided context: ### Leadership Announcement (October 2026) Novasphere Dynamics appointed **Dr. Elina Vance** as **Chief AI Ethics Officer**, overseeing autonomous safety auditing, algorithmic transparency, and EU AI Act compliance. ### Edge Memory Optimization: QuantumCache v4 Reduced resident RAM usage on NVIDIA Jetson Orin edge controllers by 30% during peak Lidar burst capture. ### Leadership Announcement (October 2026) Novasphere Dynamics appointed **Dr. Elina Vance** as **Chief AI Ethics Officer**, overseeing autonomous safety auditing, algorithmic transparency, and EU AI Act compliance.
Missing   : I don't know based on the provided context. The supplied documents do not contain enough information to answer this question.


## 5. Common Mistakes
- Treating fine-tuning as a factual database — memorized weights lag behind weekly changelogs.
- Pasting huge raw folders into prompts instead of searching for top relevant pieces first.
- Not allowing the prompt to answer `'I don't know'` when facts are absent.

## 6. Recap
- One-sentence definition: **RAG searches external documents for relevant facts and injects them into the prompt so an LLM generates grounded, up-to-date answers.**
- Standalone LLMs suffer from knowledge cutoffs, private blindness, hallucination, and context-window budgets.
- Prompt-stuffing works only on tiny single docs; RAG scales cleanly to huge libraries.

## 7. Next Notebook
Next (`01_rag_big_picture.ipynb`): Build a complete RAG pipeline in plain Python with zero libraries, then see what LangChain automates!

## 8. Video Chapter Outline
1. **Closed-Book Failure Demo** — Asking an LLM private Novasphere questions.
2. **Why Prompt-Stuffing Doesn't Scale** — Counting tokens at 5,000-file scale.
3. **Prompt-Stuffing vs Fine-Tuning vs RAG** — Decision matrix for builders.
4. **Focused Retrieval Prototype** — Trimming context down to pricing lines only.
5. **One-Sentence RAG Definition** — Summary before coding the full pipeline.